In [1]:
import pandas as pd
import numpy as np
import torch_geometric
from torch_geometric.data import Data, DataLoader
import os
import pandas as pd
import ast

d:\software\Anaconda3\envs\finnlp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 创建一个空的DataFrame
dataframe = pd.DataFrame()

# 指定文件夹路径
folder_path = 'data_raw'
news0328=pd.read_csv("data_processed/新闻_20220328.csv")
news0328.shape

(5584, 12)

In [3]:
news0328["ts_code"]

0       000001.SZ
1       000002.SZ
2       000005.SZ
3       000008.SZ
4       000048.SZ
          ...    
5579    871642.BJ
5580    871857.BJ
5581    871981.BJ
5582    872925.BJ
5583    689009.SH
Name: ts_code, Length: 5584, dtype: object

In [16]:
def normalize_code(symbol, pre_close=None):
    """
    归一化证券代码
    
    :param symbol: 如 '1' 或 '000001'
    :param pre_close: 前收盘价（用于推断指数）
    :return: 证券代码的全称 如 '000001.SZ'
    """
    if not isinstance(symbol, str):
        return symbol

    # 处理带交易所前缀的代码（如 sz000001）
    if symbol.startswith('sz') and len(symbol) == 8:
        normalized = symbol[2:8].zfill(6)
        return f"{normalized}.{EXCHANGE.SZ}"
    elif symbol.startswith('sh') and len(symbol) == 8:
        normalized = symbol[2:8].zfill(6)
        return f"{normalized}.{EXCHANGE.SH}"

    # 尝试补全为6位代码（如 '1' → '000001'）
    if len(symbol) < 6:
        symbol = symbol.zfill(6)  # 左侧补零到6位

    # 仅处理6位代码
    if len(symbol) != 6:
        return symbol  # 非6位代码直接返回

    # 规则匹配
    if symbol.startswith('00'):
        if pre_close and pre_close > 2000:  # 推断为上证指数
            return f"{symbol}.{EXCHANGE.SH}"
        else:
            return f"{symbol}.{EXCHANGE.SZ}"
    elif symbol.startswith(('399', '159', '150', '16', '184801', '201872')):
        return f"{symbol}.{EXCHANGE.SZ}"
    elif symbol.startswith(('50', '51', '60', '688', '900')) or symbol == '751038':
        return f"{symbol}.{EXCHANGE.SH}"
    elif symbol[:3] in ['000', '001', '002', '200', '300']:
        return f"{symbol}.{EXCHANGE.SZ}"
    else:
        return f"{symbol}.{EXCHANGE.SZ}"  # 默认归属深交所（根据需求调整）

In [22]:
normalize_code(1)

1

In [26]:
csi100=pd.read_csv("CSIA100.csv")
csi100["成份券代码Constituent Code"] = csi100["成份券代码Constituent Code"].astype(str).apply(normalize_code)
# 保存结果
output_path = 'CSIA100_normalized.csv'
csi100.to_csv(output_path, index=False, encoding='utf-8-sig')

In [27]:
###筛选CSI100中的股票
def filter_news_by_constituents(news_file, constituents_file, output_file=None):
    """
    筛选新闻文件中 ts_code 属于成分股代码的行
    
    Args:
        news_file (str): 新闻文件路径（如 "新闻_20220328.csv"）
        constituents_file (str): 成分股文件路径（如 "CSIA100_normalized.csv"）
        output_file (str, optional): 输出文件路径。如果为 None，则不保存
    
    Returns:
        pd.DataFrame: 筛选后的新闻数据
    """
    # 读取新闻数据
    news_df = pd.read_csv(news_file)
    
    # 读取成分股数据
    constituents_df = pd.read_csv(constituents_file)
    constituent_codes = set(constituents_df['成份券代码Constituent Code'])
    
    # 筛选新闻数据
    filtered_news = news_df[news_df['ts_code'].isin(constituent_codes)]
    
    # 可选：保存结果
    if output_file:
        filtered_news.to_csv(output_file, index=False, encoding='utf-8-sig')
        print(f"筛选结果已保存到: {output_file}")
    
    return filtered_news


In [28]:
filter_news_by_constituents("data_processed/新闻_20220328.csv", 'CSIA100_normalized.csv', output_file="news0328.csv")

筛选结果已保存到: news0328.csv


,ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount,label
1,000002.SZ,20220328,17.10,17.91,17.01,17.77,17.41,0.36,2.0678,1042085.74,1837872.774,1
8,000333.SZ,20220328,55.87,56.16,54.80,55.80,56.09,-0.29,-0.5170,215852.82,1198968.556,0
26,000651.SZ,20220328,31.06,31.64,30.75,31.64,31.52,0.12,0.3807,277666.41,866933.961,1
44,000792.SZ,20220328,31.28,31.68,30.19,31.21,31.63,-0.42,-1.3279,743099.84,2302710.335,0
82,002049.SZ,20220328,197.00,203.61,196.20,200.28,199.88,0.40,0.2001,36182.26,727835.443,1
...,...,...,...,...,...,...,...,...,...,...,...,...
5113,688012.SH,20220328,113.80,115.37,113.10,114.68,115.20,-0.52,-0.4514,26297.12,300165.656,0
5132,688036.SH,20220328,97.83,99.38,96.37,98.41,99.00,-0.59,-0.5960,15257.83,149474.910,0
5187,688111.SH,20220328,184.59,187.32,180.64,181.90,188.36,-6.46,-3.4296,16453.34,301474.943,0
5282,688256.SH,20220328,62.90,64.82,62.28,63.90,63.68,0.22,0.3455,15119.27,96417.125,1


In [35]:
unique_values=df_news0328=pd.read_csv("news0328.csv")["ts_code"].unique()
unique_values.shape

(98,)